Extract recorded HealthBench judge scores and token totals, and verify the saved criterion reference dataset...


In [1]:
from pathlib import Path
import gzip
import hashlib
import json
import re
import pandas as pd
ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'evaluations/human_and_model_validation/healthbench_judge_validation').is_dir() and (p / 'data').is_dir()))
OUT = ROOT / 'evaluations/human_and_model_validation/healthbench_judge_validation'
records = json.loads((OUT / 'recorded_outputs.json').read_text(encoding='utf-8'))
rows = []
for record in records:
    match = re.fullmatch('\\[META\\]\\s+(\\S+)\\s+->\\s+score=([0-9.]+),\\s+tokens=(\\d+)', record['recorded_output'])
    assert match is not None
    rows.append(dict(source_cell=record['source_cell'], judge_identifier=match[1], meta_examples=record['meta_examples'], macro_f1_recorded=float(match[2]), recorded_tokens=int(match[3]), included_in_article=record['included_in_article']))
runs = pd.DataFrame(rows)
assert len(runs) == 19 and runs.source_cell.is_unique
article = runs.loc[runs.included_in_article].copy()
assert len(article) == 18 and article.meta_examples.eq(120).all()
runs.to_csv(OUT / 'recorded_runs.csv', index=False)
article.to_csv(OUT / 'article_healthbench_judges.csv', index=False)
display(article)


,source_cell,judge_identifier,meta_examples,macro_f1_recorded,recorded_tokens,included_in_article
0,4,aliceai-llm/latest,120,0.670635,211832,True
1,5,yandexgpt-lite/latest,120,0.594788,207352,True
2,6,yandexgpt/latest,120,0.588783,208046,True
3,8,gpt-oss-20b/latest,120,0.647011,272884,True
4,9,gpt-oss-120b/latest,120,0.686423,240457,True
5,10,gemma-3-27b-it/latest,120,0.636400,201100,True
6,11,qwen3-235b-a22b-fp8/latest,120,0.647341,244127,True
7,12,deepseek-v32/latest,120,0.649155,453216,True
8,13,qwen3.5-35b-a3b-fp8/latest,120,0.683492,207589,True
9,15,gpt-4.1-mini,120,0.672161,198482,True


In [2]:
raw_hash = hashlib.sha256()
reference_rows = []
with gzip.open(OUT / 'data/healthbench_meta_eval.jsonl.gz', 'rb') as stream:
    for row_number, line in enumerate(stream, 1):
        raw_hash.update(line)
        item = json.loads(line)
        labels = item['binary_labels']
        assert len(labels) == len(item['anonymized_physician_ids'])
        assert all((isinstance(value, bool) for value in labels))
        reference_rows.append(dict(source_row=row_number, prompt_id=item['prompt_id'], completion_id=item['completion_id'], category=item['category'], rubric_sha256=hashlib.sha256(item['rubric'].encode('utf-8')).hexdigest(), physician_ratings=len(labels), positive_ratings=sum(labels), negative_ratings=len(labels) - sum(labels)))
reference = pd.DataFrame(reference_rows)
provenance = json.loads((OUT / 'provenance.json').read_text(encoding='utf-8'))
assert raw_hash.hexdigest() == provenance['raw_dataset']['uncompressed_sha256']
assert len(reference) == 29511
reference.to_csv(OUT / 'reference_items.csv', index=False)
summary = dict(reference_items=len(reference), prompt_ids=int(reference.prompt_id.nunique()), completion_ids=int(reference.completion_id.nunique()), physician_ratings=int(reference.physician_ratings.sum()), raw_sha256=raw_hash.hexdigest(), recorded_runs=len(runs), article_runs=len(article), individual_model_predictions_available=False, exact_120_item_indices_available=False, macro_f1_recomputed_from_predictions=False)
(OUT / 'evidence_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
display(pd.Series(summary))


reference_items                                                                       29511
prompt_ids                                                                             3671
completion_ids                                                                        14592
physician_ratings                                                                     60896
raw_sha256                                6a343ed3fcb6f28353c2c6c157ae30ddf2db874c947b27...
recorded_runs                                                                            19
article_runs                                                                             18
individual_model_predictions_available                                                False
exact_120_item_indices_available                                                      False
macro_f1_recomputed_from_predictions                                                  False
dtype: object

Reconstructed evaluator selection


In [3]:
import gzip
import random
import hashlib
import csv
from pathlib import Path
ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'evaluations/human_and_model_validation/healthbench_judge_validation/data/healthbench_meta_eval.jsonl.gz').is_file()))
OUT = ROOT / 'evaluations/human_and_model_validation/healthbench_judge_validation'
with gzip.open(OUT / 'data/healthbench_meta_eval.jsonl.gz', 'rt', encoding='utf-8') as stream:
    reference = [dict(json.loads(line), _source_row=i + 1) for i, line in enumerate(stream)]
selected = random.Random(0).sample(reference, 120)
selection = [{'source_row': row['_source_row'], 'prompt_id': row['prompt_id'], 'completion_id': row['completion_id'], 'category': row['category'], 'rubric_sha256': hashlib.sha256(row['rubric'].encode('utf-8')).hexdigest()} for row in selected]
with (OUT / 'selected_120_reconstructed.csv').open('w', encoding='utf-8-sig', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(selection[0]))
    writer.writeheader()
    writer.writerows(selection)
assert len(selection) == len({row['source_row'] for row in selection}) == 120
summary = json.loads((OUT / 'evidence_summary.json').read_text(encoding='utf-8'))
summary['selected_120_reconstructed'] = True
summary['historical_individual_predictions_available'] = False
(OUT / 'evidence_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print("120 unique reference items reconstructed from the saved evaluator's seed; historical predictions unavailable.")


120 unique reference items reconstructed from the saved evaluator's seed; historical predictions unavailable.
